# Imaginary-time density memory: flattened-parent ground state

One deterministic 20×30 hard-wall ground state, not a stochastic circuit campaign.
The signed OW-projector parent retains its wall dispersion; no additional spectral flattening.

For a quadratic density observable, evaluate
$C_O(\tau)=\sum_{a\in\mathrm{occ},b\in\mathrm{empty}}|O_{ba}|^2e^{-\tau(\epsilon_b-\epsilon_a)}$.
Local density is summed over the two orbitals and averaged over origins y.
Column charge sums all y before forming the correlator and is divided by Ny.
Imaginary time is in inverse parent-energy units, not circuit cycles.
The ground state and propagation use the same regulated parent (occupation twist 1e-7).
No error bars or power-law fits are imposed. See `theory.tex` for conventions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# All settings. CPU is sufficient; no A100 required.
REPORT_ONLY = False
CPU_THREADS = 2
# Scientific settings are visible but locked to this named revision.
CONFIG = {'revision': 'flattened_imaginary_time_density_nx20_ny30_hard_v1',
 'Nx': 20,
 'Ny': 30,
 'alpha_1': 1.0,
 'alpha_2': 30.0,
 'nshell': 1,
 'DW': True,
 'dw_truncation': True,
 'trial_orbitals': 'X',
 'filling_fraction': 0.5,
 'dtype': 'complex128',
 'occupation_twist': 1e-07,
 'tau_min': 0.001,
 'tau_max': 1000.0,
 'tau_points': 240,
 'tau_chunk': 8,
 'normalization_tolerance': 1e-12}
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/24_flattened_imaginary_time_density')
LOCAL_BUNDLE_DIR = Path('/content/24_flattened_imaginary_time_density')
SCRATCH_ROOT = Path('/content/imaginary_time_density_scratch')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['revision']


## Stage locally and run/resume

The one completed result is verified before skipping. An interrupted task restarts; there are no trajectory or state histories. Progress counts imaginary times. DriveFS readback is not an independent server-side check.

In [ ]:
import codecs
import json
import shutil
import subprocess
import sys

LOCAL_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
for relative in ['run_campaign.py', 'density_correlations.py', 'plot_results.py', 'src/classA_U1FGTN.py', 'src/occupied_frame.py']:
    destination = LOCAL_BUNDLE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_DRIVE_DIR / relative, destination)
config_path = LOCAL_BUNDLE_DIR / 'resolved_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR / 'run_campaign.py'),
           '--config', str(config_path), '--output-root', str(OUTPUT_ROOT),
           '--scratch-root', str(SCRATCH_ROOT), '--threads', str(CPU_THREADS)]
if REPORT_ONLY:
    command.append('--report-only')
print('[launch]', ' '.join(command), flush=True)
decoder = codecs.getincrementaldecoder('utf-8')('replace')
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0) as process:
    while True:
        chunk = process.stdout.read(4096)
        if not chunk:
            break
        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
    sys.stdout.write(decoder.decode(b'', final=True)); sys.stdout.flush()
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)


## Read verified data

Each plot has its own editable cell. Earlier results are never pooled.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
sys.path.insert(0, str(LOCAL_BUNDLE_DIR))
from run_campaign import identity, paths, verified
from plot_results import configure_style, draw_curves
configure_style()
DATA = None
ident = identity(CONFIG)
if verified(OUTPUT_ROOT, ident):
    result_path, receipt_path = paths(OUTPUT_ROOT, ident)
    with np.load(result_path, allow_pickle=False) as saved:
        DATA = {key: saved[key] for key in saved.files}
    print(json.loads(str(DATA['diagnostics_json'])))
else:
    print('No verified result yet; plots skipped.')


## Local loglog

Raw connected autocorrelation; zeros are omitted on logarithmic axes.

In [ ]:
if DATA is not None:
    fig, ax = plt.subplots(figsize=(3.375, 2.6), layout='constrained')
    draw_curves(ax, DATA, kind='local', log_time=True)
    plt.show()


## Local semilog

Raw connected autocorrelation; zeros are omitted on logarithmic axes.

In [ ]:
if DATA is not None:
    fig, ax = plt.subplots(figsize=(3.375, 2.6), layout='constrained')
    draw_curves(ax, DATA, kind='local', log_time=False)
    plt.show()


## Column loglog

Raw connected autocorrelation; zeros are omitted on logarithmic axes.

In [ ]:
if DATA is not None:
    fig, ax = plt.subplots(figsize=(3.375, 2.6), layout='constrained')
    draw_curves(ax, DATA, kind='column', log_time=True)
    plt.show()


## Spatially resolved memory

Normalize by each x slice’s own equal-time variance; undefined ratios are masked.

In [ ]:
if DATA is not None:
    fig, ax = plt.subplots(figsize=(3.375, 2.6), layout='constrained')
    values = np.ma.masked_invalid(DATA['local_normalized'][1:])
    values = np.ma.masked_less_equal(values, 0)
    image = ax.pcolormesh(DATA['x'], DATA['tau'][1:], values, shading='nearest', cmap='Blues', norm=LogNorm(1e-12, 1))
    ax.set(yscale='log', xlabel='$x$', ylabel=r'imaginary time $\tau$')
    fig.colorbar(image, ax=ax, label=r'$C_x(\tau)/C_x(0)$', extend='min')
    plt.show()


## Raw diagnostics

No fitted exponent, no independent samples, no sampling uncertainty.

In [ ]:
if DATA is not None:
    print('Local normalization valid:', DATA['local_normalization_valid'])
    print('Column normalization valid:', DATA['column_normalization_valid'])
    print('Diagnostics:', json.loads(str(DATA['diagnostics_json'])))


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
